# 6. Monte Carlo experiment (Table 8)

Data are simulated from four DGPs calibrated to the full-sample estimates (Gaussian, Merton, GARCH-t, GARCH-Merton). Every model is fitted on `n_train` observations and scored on the next `n_test`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / 'config.toml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import json
import pandas as pd
from IPython.display import display, Image
from mxntail import pipeline, reporting
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
cfg = pipeline.load_config()
RES = pipeline.results_dir()

def cached(stage, output):
    """Run a pipeline stage only if its output file is missing."""
    if not (RES / output).exists():
        print(f'Running stage {stage!r} (this can take a while)...')
        pipeline.STAGES[stage](cfg)
    else:
        print(f'Using cached results/{output}. Delete it to recompute.')


In [2]:
cached('montecarlo', 'montecarlo_scores.csv')
print(json.loads((RES/'montecarlo_dgp_parameters.json').read_text()))

Using cached results/montecarlo_scores.csv. Delete it to recompute.
{'Gaussian': [0.011283137948579464, 0.6876600889027326], 'Merton': [-0.04388901539649468, 0.41851668228542255, 0.30558028618630523, 0.18054923852588328, 0.9344486335566652], 'GARCH-t': [-0.01943900266997155, 0.008844030960210308, 0.12755403680760719, 0.8585263048300171, 6.585812287203566], 'GARCH-skew-t': [-0.004079056077779853, 0.00799805792011343, 0.12021275214839243, 0.8657792109917799, 7.035087364695981, 0.15393757041875744], 'GARCH-Merton': [-0.041924011891594906, 0.004429801534582855, 0.10832976882312012, 0.8510118139232883, 0.1350736689599139, 0.30440215253898756, 0.7061607314511396]}


In [3]:
num = reporting.Numbers(); reporting.TABLES.mkdir(parents=True, exist_ok=True)
reporting.table_montecarlo(cfg, num)
m = pd.read_csv(RES/'montecarlo_summary.csv')
m.pivot(index='dgp', columns='model', values='mean_nls').round(4)

model,GARCH-Kou,GARCH-Merton,GARCH-skew-t,GARCH-t,Gaussian,Kou,Merton,Student-t
dgp,,,,,,,,
GARCH-Merton,0.8635,0.8637,0.8682,0.8704,0.9890,0.9403,0.9403,0.9415
GARCH-skew-t,0.8497,0.8519,0.8412,0.8472,1.1582,0.9632,0.9732,0.9586
GARCH-t,0.8715,0.8721,0.8648,0.8644,1.1707,0.9829,0.9902,0.9765
Gaussian,1.0481,1.0482,1.0484,1.0480,1.0477,1.0479,1.0481,1.0478
Merton,0.9427,0.9423,0.9452,0.9477,1.0232,0.9424,0.9420,0.9474


In [4]:
m.pivot(index='dgp', columns='model', values='share_best').round(2)

model,GARCH-Kou,GARCH-Merton,GARCH-skew-t,GARCH-t,Gaussian,Kou,Merton,Student-t
dgp,,,,,,,,
GARCH-Merton,0.44,0.44,0.08,0.04,0.00,0.00,0.00,0.00
GARCH-skew-t,0.06,0.06,0.77,0.11,0.00,0.00,0.00,0.00
GARCH-t,0.08,0.08,0.34,0.50,0.00,0.00,0.00,0.00
Gaussian,0.10,0.08,0.16,0.14,0.24,0.02,0.16,0.12
Merton,0.13,0.26,0.07,0.03,0.00,0.16,0.30,0.04


In [5]:
rec = pd.read_csv(RES/'montecarlo_recovery.csv')
rec[rec.dgp=='GARCH-Merton'].filter(like='lambda').describe().round(3)

,hat_lambda,true_lambda
count,200.000,200.000
mean,0.141,0.135
std,0.073,0.000
min,0.023,0.135
25%,0.094,0.135
50%,0.130,0.135
75%,0.179,0.135
max,0.678,0.135
